# 同花顺财务数据处理流水线 — 审计追踪

本 Notebook 完整展示从**同花顺 PC 端原始 XLS 文件**到**标准化财务指标 / DCF 估值**的每一步处理过程，
方便逐步核验数据是否正确。

| 步骤 | 对应脚本 | 说明 |
|------|----------|------|
| 1 | `01_RoyalFlushData2csv_10years.py` | 原始 XLS → 清洗后 CSV |
| 2 | `02_CheckStatements.py` | 三表一致性检验 |
| 3 | `03_ExtractCalc.py` / `04_FinancialCoreMetrics.py` | 核心指标提取 |
| 4 | `Rebuild_BS/PL/CF.py` | 报表标准化重构 |
| 5 | `Generate_DCF_Valuation.py` | DCF 估值模型 |

## 0. 环境准备

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

import xlrd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from IPython.display import display, HTML

matplotlib.rcParams['font.family'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, str(Path('.').resolve()))
from pipeline_utils import validate_rawdata, ensure_output_dirs, CSV_DIR, RAW_DIR

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.max_rows', 60)

ensure_output_dirs()
ticker, raw_files = validate_rawdata()

print(f'股票代码 : {ticker}')
print('原始文件 :')
for k, v in raw_files.items():
    print(f'  [{k}]  {v}')

---
## 1. 原始 XLS 数据预览（同花顺导出，未做任何处理）

以资产负债表为例，展示 xlrd 直接读取的原始内容，包括：
- 同花顺标头行（含版权声明）
- 原始科目名（含 `*`、BOM、`.0` 浮点年份等噪声）
- `--` 占位符（代表无数据）

In [ ]:
def _raw_preview(xls_path, n_rows=25):
    book = xlrd.open_workbook(str(xls_path), ignore_workbook_corruption=True)
    sheet = book.sheet_by_index(0)
    rows = [sheet.row_values(i) for i in range(min(n_rows + 3, sheet.nrows))]
    header = rows[1]
    header_clean = [str(h).replace('.0', '') if str(h).endswith('.0') else str(h) for h in header]
    data = rows[2:]
    df = pd.DataFrame(data, columns=header_clean[:len(data[0])] if data else header_clean)
    return df

for label, path in raw_files.items():
    if label == 'price':
        continue
    name_map = {'balance_sheet': '资产负债表', 'profit_loss': '利润表', 'cash_flow': '现金流量表'}
    print(f'\n【{name_map.get(label, label)}】原始 XLS 前 20 行 — {path.name}')
    df_raw = _raw_preview(path, n_rows=20)
    display(
        df_raw.head(20)
        .style
        .set_table_styles([{'selector': 'th', 'props': [('background-color', '#f0f0f0')]}])
        .set_properties(**{'font-size': '12px'})
        .highlight_null(color='#ffe0e0')
    )

---
## 2. 第一步：XLS → CSV 转换

对应脚本：`01_RoyalFlushData2csv_10years.py`

**清洗操作：**
1. 去掉同花顺标头行（第 0 行），使用第 1 行作为真正的表头
2. 以科目名作为 index
3. `--` → `0`
4. 列名去除 `.0` 浮点后缀（年份列：`2023.0` → `2023`）
5. 保留最近 10 年并按时间正序排列
6. 删除全空行

In [ ]:
def _trans_csv_audit(xls_path):
    """复现 trans_csv 的每一步，并返回各阶段 DataFrame 供对比。"""
    stages = {}

    # 原始读取
    book = xlrd.open_workbook(str(xls_path), ignore_workbook_corruption=True)
    sheet = book.sheet_by_index(0)
    data = [sheet.row_values(i) for i in range(sheet.nrows)]

    df0 = pd.DataFrame(data[1:], columns=data[1])
    df0 = df0.iloc[1:, ]
    stages['① 去标头行后（含 -- 和浮点年份）'] = df0.copy()

    # index 设置
    df1 = df0.copy()
    df1.index = df1.iloc[:, 0]
    df1 = df1.iloc[:, 1:]
    stages['② 科目设为 index'] = df1.copy()

    # -- → 0
    df2 = df1.copy()
    df2.replace('--', 0, inplace=True)
    stages['③ -- 替换为 0'] = df2.copy()

    # 列名清洗
    df3 = df2.copy()
    df3.columns = df3.columns.map(lambda x: str(x).replace('.0', ''))
    stages['④ 年份列名去 .0 后缀'] = df3.copy()

    # dropna
    df4 = df3.copy()
    for c in df4.columns:
        df4[c] = pd.to_numeric(df4[c], errors='coerce')
    df4.dropna(axis=0, how='any', inplace=True)
    stages['⑤ 删除全空行、转数值'] = df4.copy()

    # 最近 10 年，正序
    df5 = df4.iloc[:, :10]
    df5 = df5[df5.columns[::-1]]
    stages['⑥ 保留最近 10 年并正序排列（最终输出）'] = df5.copy()

    return stages


for label, path in list(raw_files.items())[:1]:          # 以资产负债表为例
    print(f'\n=== 资产负债表转换过程 ===')
    stages = _trans_csv_audit(path)
    for stage_name, df_stage in stages.items():
        print(f'\n{stage_name}')
        display(
            df_stage.head(8)
            .style
            .set_properties(**{'font-size': '11px'})
            .highlight_null(color='#ffe0e0')
        )

In [ ]:
# 实际执行转换，写出 CSV
import subprocess
result = subprocess.run([sys.executable, '01_RoyalFlushData2csv_10years.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

---
## 3. 第二步：三表一致性检验

对应脚本：`02_CheckStatements.py`

共 5 项校验：

| # | 检验项 | 公式 |
|---|--------|------|
| 1 | 资产负债表平衡 | 资产合计 = 负债合计 + 所有者权益合计 |
| 2 | 现金流量表平衡 | 净增加额 = CFO + CFI + CFF + 汇率影响 |
| 3 | 现金滚动校验 | 期初 + 净增加额 = 期末 |
| 4 | 货币资金对比 | 观察性，不强等式 |
| 5 | 净利润与权益变动 | 同向性判断 |

In [ ]:
BS_CSV = './results/csv/bs.csv'
PL_CSV = './results/csv/pl.csv'
CF_CSV = './results/csv/cf.csv'


def _load(path):
    df = pd.read_csv(path)
    df.rename(columns={df.columns[0]: '科目'}, inplace=True)
    df['科目'] = df['科目'].astype(str).str.strip()
    df.columns = ['科目'] + [str(c).strip() for c in df.columns[1:]]
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df.set_index('科目')


def _find(df, candidates):
    for c in candidates:
        if c in df.index:
            return df.loc[c]
    return None


bs = _load(BS_CSV)
pl = _load(PL_CSV)
cf = _load(CF_CSV)
years = [c for c in bs.columns if c in cf.columns and c in pl.columns]
print(f'三表共同覆盖年份：{years}')


def _style_check(df, bool_col):
    def color_row(row):
        val = row.get(bool_col, True)
        color = '#d4edda' if bool(val) else '#f8d7da'
        return [f'background-color: {color}'] * len(row)
    return df.style.apply(color_row, axis=1).format('{:,.0f}', subset=[c for c in df.columns if df[c].dtype in ['float64', 'int64']], na_rep='NaN')


# ---- 检验 1：资产负债表平衡 ----
assets  = _find(bs, ['*资产合计(元)', '资产总计(元)', '资产合计(元)'])
liabs   = _find(bs, ['*负债合计(元)', '负债合计(元)'])
equity  = _find(bs, ['*所有者权益（或股东权益）合计(元)', '所有者权益合计(元)', '所有者权益（或股东权益）合计(元)'])

check1 = pd.DataFrame({
    '资产合计':   assets[years].values,
    '负债合计':   liabs[years].values,
    '所有者权益': equity[years].values,
    '负债+权益':  (liabs[years] + equity[years]).values,
    '差额':       (assets[years] - liabs[years] - equity[years]).values,
}, index=years)
check1['通过'] = np.isclose(check1['差额'], 0, atol=1e3)

print('\n【检验 1】资产负债表平衡')
display(_style_check(check1, '通过'))

In [ ]:
# ---- 检验 2：现金流量表平衡 ----
cfo       = _find(cf, ['*经营活动产生的现金流量净额(元)', '经营活动产生的现金流量净额(元)'])
cfi       = _find(cf, ['*投资活动产生的现金流量净额(元)', '投资活动产生的现金流量净额(元)'])
cff       = _find(cf, ['*筹资活动产生的现金流量净额(元)', '筹资活动产生的现金流量净额(元)'])
net_cash  = _find(cf, ['*现金及现金等价物净增加额(元)', '五、现金及现金等价物净增加额(元)', '现金及现金等价物净增加额(元)'])
fx_row    = _find(cf, ['四、汇率变动对现金及现金等价物的影响(元)', '汇率变动对现金及现金等价物的影响(元)'])
fx        = fx_row[years] if fx_row is not None else pd.Series(0.0, index=years)

check2 = pd.DataFrame({
    'CFO': cfo[years].values, 'CFI': cfi[years].values, 'CFF': cff[years].values,
    '汇率影响': fx.values,
    '理论净增加额': (cfo[years] + cfi[years] + cff[years] + fx).values,
    '报表净增加额': net_cash[years].values,
    '差额': (net_cash[years] - cfo[years] - cfi[years] - cff[years] - fx).values,
}, index=years)
check2['通过'] = np.isclose(check2['差额'], 0, atol=1e3)

print('\n【检验 2】现金流量表平衡')
display(_style_check(check2, '通过'))


# ---- 检验 3：现金滚动 ----
begin_cash  = _find(cf, ['加：期初现金及现金等价物余额(元)', '期初现金及现金等价物余额(元)'])
ending_cash = _find(cf, ['*期末现金及现金等价物余额(元)', '六、期末现金及现金等价物余额(元)', '期末现金及现金等价物余额(元)'])

if begin_cash is not None:
    check3 = pd.DataFrame({
        '期初现金':   begin_cash[years].values,
        '净增加额':   net_cash[years].values,
        '理论期末':   (begin_cash[years] + net_cash[years]).values,
        '报表期末':   ending_cash[years].values,
        '差额':       (ending_cash[years] - begin_cash[years] - net_cash[years]).values,
    }, index=years)
    check3['通过'] = np.isclose(check3['差额'], 0, atol=1e3)
    print('\n【检验 3】现金滚动校验')
    display(_style_check(check3, '通过'))

In [ ]:
# ---- 检验 4：货币资金 vs 期末现金对比 ----
cash_bs = _find(bs, ['货币资金(元)', '货币资金'])
if cash_bs is not None:
    check4 = pd.DataFrame({
        '资产负债表_货币资金':     cash_bs[years].values,
        '现金表_期末现金等价物':   ending_cash[years].values,
        '差额（非强等式，供参考）': (cash_bs[years] - ending_cash[years]).values,
    }, index=years)
    print('\n【检验 4】货币资金 vs 期末现金（观察性，差额可由受限资金等解释）')
    display(check4.style.format('{:,.0f}', na_rep='NaN').bar(subset=['差额（非强等式，供参考）'], align='zero', color=['#f28b82', '#81c995']))


# ---- 检验 5：归母净利润与权益变动同向性 ----
parent_np     = _find(pl, ['*归属于母公司所有者的净利润(元)', '归属于母公司所有者的净利润(元)'])
parent_equity = _find(bs, ['*归属于母公司所有者权益合计(元)', '归属于母公司所有者权益合计(元)'])

if parent_np is not None and parent_equity is not None:
    eq_chg = parent_equity[years].astype(float).diff()
    check5 = pd.DataFrame({
        '归母净利润':   parent_np[years].values,
        '归母权益增加额': eq_chg.values,
    }, index=years)
    check5['同向性'] = np.where(
        (check5['归母净利润'] >= 0) & (check5['归母权益增加额'].fillna(0) >= 0), '✓ 大体一致', '⚠ 需关注分红/回购/OCI'
    )
    print('\n【检验 5】归母净利润与归母权益变动联动')
    display(check5.style.format('{:,.0f}', subset=['归母净利润', '归母权益增加额'], na_rep='NaN'))

---
## 4. 第三步：核心指标提取与计算

对应脚本：`04_FinancialCoreMetrics.py`

展示每个科目的**匹配结果**（matched / missing），然后展示计算后的关键指标。

In [ ]:
def _load_item(path):
    df = pd.read_csv(path)
    df.rename(columns={df.columns[0]: 'Item'}, inplace=True)
    df['Item'] = df['Item'].astype(str).str.replace('\ufeff', '', regex=False).str.strip()
    df.columns = ['Item'] + [str(c).strip() for c in df.columns[1:]]
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df.set_index('Item')


pl2 = _load_item(PL_CSV)
bs2 = _load_item(BS_CSV)
cf2 = _load_item(CF_CSV)
yrs = sorted(set(pl2.columns) & set(bs2.columns) & set(cf2.columns))

ITEM_MAP = {
    '营业收入':      (pl2, ['其中：营业收入(元)', '营业收入(元)']),
    '营业成本':      (pl2, ['其中：营业成本(元)', '营业成本(元)']),
    '营业税金及附加': (pl2, ['营业税金及附加(元)']),
    '销售费用':      (pl2, ['销售费用(元)']),
    '管理费用':      (pl2, ['管理费用(元)']),
    '研发费用':      (pl2, ['研发费用(元)']),
    '营业利润':      (pl2, ['三、营业利润(元)', '*三、营业利润(元)', '营业利润(元)']),
    '利息费用':      (pl2, ['其中：利息费用(元)', '利息费用(元)']),
    '利息收入':      (pl2, ['利息收入(元)', '其中：利息收入(元)']),
    '利润总额':      (pl2, ['四、利润总额(元)', '*四、利润总额(元)', '利润总额(元)']),
    '所得税':        (pl2, ['减：所得税费用(元)', '所得税费用(元)']),
    '净利润':        (pl2, ['五、净利润(元)', '*五、净利润(元)', '净利润(元)']),
    '归母净利润':    (pl2, ['归属于母公司所有者的净利润(元)', '*归属于母公司所有者的净利润(元)']),
    '扣非净利润':    (pl2, ['扣除非经常性损益后的净利润(元)']),
    '折旧':          (cf2, ['固定资产折旧、油气资产折耗、生产性生物资产折旧(元)', '固定资产折旧、油气资产折耗、生产性生物资产折旧']),
    '无形摊销':      (cf2, ['无形资产摊销(元)', '无形资产摊销']),
    'CapEX':         (cf2, ['购建固定资产、无形资产和其他长期资产支付的现金(元)']),
    '处置长期资产现金': (cf2, ['处置固定资产、无形资产和其他长期资产收回的现金净额(元)']),
    'CFO':           (cf2, ['*经营活动产生的现金流量净额(元)', '经营活动产生的现金流量净额(元)']),
    'CFI':           (cf2, ['*投资活动产生的现金流量净额(元)', '投资活动产生的现金流量净额(元)']),
    'CFF':           (cf2, ['*筹资活动产生的现金流量净额(元)', '筹资活动产生的现金流量净额(元)']),
    '归母权益':      (bs2, ['归属于母公司所有者权益合计(元)', '*归属于母公司所有者权益合计(元)']),
    '资产总计':      (bs2, ['资产总计(元)', '*资产总计(元)', '*资产合计(元)', '资产合计(元)']),
    '负债合计':      (bs2, ['负债合计(元)', '*负债合计(元)']),
}

match_log = []
raw_data = {}
for label, (df_src, candidates) in ITEM_MAP.items():
    hit = next((c for c in candidates if c in df_src.index), None)
    match_log.append({'科目': label, '状态': '✓ 匹配' if hit else '✗ 未找到', '匹配原始科目名': hit or '—'})
    raw_data[label] = df_src.loc[hit, yrs].astype(float) if hit else pd.Series(np.nan, index=yrs)

match_df = pd.DataFrame(match_log)
print('科目匹配情况')

def _color_status(val):
    return 'color: green; font-weight: bold' if '✓' in str(val) else 'color: red; font-weight: bold'

display(match_df.style.applymap(_color_status, subset=['状态']))

In [ ]:
def sd(a, b): return a.subtract(b, fill_value=0)
def sa(*s):   return sum(x.fillna(0) for x in s)
def sdiv(a, b): return a / b.replace(0, np.nan)

rv  = raw_data['营业收入']
cgs = raw_data['营业成本']
gp  = sd(rv, cgs)
opex = sa(raw_data['营业税金及附加'], raw_data['销售费用'], raw_data['管理费用'], raw_data['研发费用'])
da  = sa(raw_data['折旧'], raw_data['无形摊销'])
ebit_formula = gp - opex
ebit_check   = raw_data['营业利润'].fillna(0) + raw_data['利息费用'].fillna(0) - raw_data['利息收入'].fillna(0)
ebitda = ebit_formula + da
cfo_v  = raw_data['CFO']
capex  = raw_data['CapEX']
fcf    = cfo_v.fillna(0) - capex.fillna(0)
net_p  = raw_data['净利润']
avg_eq = (raw_data['归母权益'].shift(1) + raw_data['归母权益']) / 2
roe    = sdiv(raw_data['归母净利润'], avg_eq)

metrics = pd.DataFrame({
    'Revenue':        rv,
    'Gross Profit':   gp,
    'Opex':           opex,
    'EBIT (建模法)':   ebit_formula,
    'EBIT (报表法)':   ebit_check,
    'D&A':            da,
    'EBITDA':         ebitda,
    'Net profit':     net_p,
    'CFO':            cfo_v,
    'CapEX':          capex,
    'FCF':            fcf,
    'Gross Margin':   sdiv(gp, rv),
    'Net Margin':     sdiv(net_p, rv),
    'EBITDA Margin':  sdiv(ebitda, rv),
    'ROE':            roe,
    '资产负债率':      sdiv(raw_data['负债合计'], raw_data['资产总计']),
    'CFO/Net profit': sdiv(cfo_v, net_p),
}, index=yrs).T

print('\n核心指标汇总（亿元 / 比率）')
margin_rows = ['Gross Margin', 'Net Margin', 'EBITDA Margin', 'ROE', '资产负债率', 'CFO/Net profit']

def _fmt(val, row_name):
    if pd.isna(val): return 'NaN'
    if row_name in margin_rows: return f'{val:.1%}'
    return f'{val/1e8:,.2f} 亿'

display(
    metrics.style
    .format(lambda x: _fmt(x, ''), na_rep='NaN')
    .bar(subset=[c for c in metrics.columns], align='zero', color=['#f28b82', '#81c995'])
    .set_caption('数值单位：绝对数为亿元，比率为百分比')
)

---
## 5. 第四步：财务报表标准化重构（BS / PL / CF）

对应脚本：`Rebuild_BS.py` / `Rebuild_PL.py` / `Rebuild_CF.py`

展示：
1. 标准化前（清洗后的 CSV，科目名口径参差）
2. 标准化后（按 Standard Line Item 汇总、英文标签）
3. 每张表内置的一致性验证

In [ ]:
# 运行三张重构脚本
for script in ['Rebuild_BS.py', 'Rebuild_PL.py', 'Rebuild_CF.py']:
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    status = '✓ 完成' if r.returncode == 0 else f'✗ 失败\n{r.stderr[:300]}'
    print(f'[{script}]  {status}')

In [ ]:
# 展示重构前后对比
_rebuild_paths = {
    '资产负债表': {
        'before': './results/csv/bs.csv',
        'after_std': './results/BS_rebuilt_output/2_standardized_bs.csv',
        'check':  './results/BS_rebuilt_output/1_preprocess_checks.csv',
    },
    '利润表': {
        'before': './results/csv/pl.csv',
        'after_std': './results/PL_rebuilt_output/2_standardized_pl.csv',
        'check':  './results/PL_rebuilt_output/1_preprocess_checks.csv',
    },
    '现金流量表': {
        'before': './results/csv/cf.csv',
        'after_std': './results/CF_rebuilt_output/2_standardized_cf.csv',
        'check':  './results/CF_rebuilt_output/1_preprocess_checks.csv',
    },
}

for name, paths in _rebuild_paths.items():
    print(f'\n{'='*60}')
    print(f'{name} — 标准化前（原始科目，前 10 行）')
    before_df = pd.read_csv(paths['before'], index_col=0)
    display(before_df.head(10).style.set_properties(**{'font-size': '11px'}))

    after_path = Path(paths['after_std'])
    if after_path.exists():
        after_df = pd.read_csv(after_path)
        print(f'\n{name} — 标准化后（Standard Line Item 汇总，全部行）')
        display(
            after_df.style
            .set_properties(**{'font-size': '11px'})
            .highlight_null(color='#ffe0e0')
        )

    check_path = Path(paths['check'])
    if check_path.exists():
        chk = pd.read_csv(check_path)
        print(f'\n{name} — 内置一致性验证')
        bool_col = next((c for c in chk.columns if chk[c].dtype == bool or set(chk[c].dropna().unique()) <= {True, False}), None)
        if bool_col:
            display(_style_check(chk.set_index(chk.columns[0]), bool_col))
        else:
            display(chk.style.format('{:,.0f}', na_rep='NaN', subset=chk.select_dtypes('number').columns))

---
## 6. 趋势图

直观展示营收、利润、现金流及利润率历史趋势。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'{ticker}  财务核心趋势', fontsize=14, fontweight='bold')

x = [str(y) for y in yrs]

def _b(series): return [v / 1e8 for v in series.values]   # 亿元

ax = axes[0, 0]
ax.bar(x, _b(rv),  label='Revenue', color='#4a90d9')
ax.bar(x, _b(gp),  label='Gross Profit', color='#81c995')
ax.bar(x, _b(net_p), label='Net Profit', color='#f28b82', alpha=0.9)
ax.set_title('Revenue / Gross Profit / Net Profit（亿元）')
ax.legend(); ax.tick_params(axis='x', rotation=45)

ax = axes[0, 1]
ax.bar(x, _b(cfo_v), label='CFO', color='#4a90d9')
ax.bar(x, _b(fcf),   label='FCF', color='#81c995', alpha=0.85)
ax.set_title('CFO / FCF（亿元）')
ax.legend(); ax.tick_params(axis='x', rotation=45)

ax = axes[1, 0]
gm = sdiv(gp, rv).values * 100
nm = sdiv(net_p, rv).values * 100
em = sdiv(ebitda, rv).values * 100
ax.plot(x, gm, 'o-', label='Gross Margin', color='#4a90d9')
ax.plot(x, nm, 's-', label='Net Margin',   color='#f28b82')
ax.plot(x, em, '^-', label='EBITDA Margin', color='#f9a825')
ax.set_title('利润率趋势（%）')
ax.legend(); ax.tick_params(axis='x', rotation=45)

ax = axes[1, 1]
roe_pct = roe.values * 100
debt_ratio = sdiv(raw_data['负债合计'], raw_data['资产总计']).values * 100
ax.plot(x, roe_pct,    'o-', label='ROE (%)',       color='#7b61ff')
ax.plot(x, debt_ratio, 's--', label='资产负债率 (%)', color='#fb8c00')
ax.set_title('ROE vs 资产负债率（%）')
ax.legend(); ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('./results/trend_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存至 ./results/trend_charts.png')

---
## 7. 第五步：DCF 估值

对应脚本：`Generate_DCF_Valuation.py`

运行后展示估值模型关键假设与结果。

In [ ]:
r = subprocess.run([sys.executable, 'Generate_DCF_Valuation.py'], capture_output=True, text=True)
print(r.stdout[:2000])
if r.returncode != 0:
    print('ERROR:', r.stderr[:800])

In [ ]:
dcf_path = Path('./results/valuation_output/DCF_valuation_model.xlsx')
if dcf_path.exists():
    xl = pd.ExcelFile(dcf_path)
    print('DCF 模型工作表：', xl.sheet_names)
    for sheet in xl.sheet_names[:4]:
        print(f'\n--- {sheet} ---')
        df_s = xl.parse(sheet, index_col=0)
        display(df_s.head(20).style.set_properties(**{'font-size': '11px'}))
else:
    print('DCF 输出文件未找到，请先确认 Rebuild_*.py 均已成功运行。')

---
## 8. 全流程汇总

一键确认所有输出文件均已生成。

In [ ]:
expected_outputs = [
    ('Step 1 — XLS→CSV',             './results/csv/bs.csv'),
    ('Step 1 — XLS→CSV',             './results/csv/pl.csv'),
    ('Step 1 — XLS→CSV',             './results/csv/cf.csv'),
    ('Step 1 — XLS→CSV',             './results/csv/price.csv'),
    ('Step 2 — 三表检验',             './results/三表一致性检验结果.xlsx'),
    ('Step 3 — 核心指标',             './results/financial_core_metrics_plus.xlsx'),
    ('Step 3 — Markdown 报告',        './results/financial_core_metrics_report.md'),
    ('Step 4 — BS 重构',              './results/BS_rebuilt_output/2_standardized_bs.csv'),
    ('Step 4 — PL 重构',              './results/PL_rebuilt_output/2_standardized_pl.csv'),
    ('Step 4 — CF 重构',              './results/CF_rebuilt_output/2_standardized_cf.csv'),
    ('Step 5 — DCF 估值',             './results/valuation_output/DCF_valuation_model.xlsx'),
    ('Trend Charts',                  './results/trend_charts.png'),
]

rows = []
for step, path in expected_outputs:
    p = Path(path)
    rows.append({'Step': step, '文件': path, '状态': '✓ 存在' if p.exists() else '✗ 缺失', '大小': f'{p.stat().st_size/1024:.1f} KB' if p.exists() else '—'})

summary = pd.DataFrame(rows)

def _color_status2(val):
    return 'color: green; font-weight: bold' if '✓' in str(val) else 'color: red; font-weight: bold'

print('全流程输出文件检查：')
display(summary.style.applymap(_color_status2, subset=['状态']))